## PREPARING DATA

In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from matplotlib.ticker import MultipleLocator


In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent
    print(script_dir)
    data_folder = script_dir /'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputDataRaw = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIGraw = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
timeSeriesData_BIGraw = timeSeriesData_BIGraw.set_index("seconds",drop=False)

In [ ]:
a = userInputDataRaw.index
b = timeSeriesData_BIGraw["keys"].unique()
diff_all = list(set(a).symmetric_difference(set(b)))
print(diff_all)  
userInputDataRaw.index = timeSeriesData_BIGraw["keys"].unique()
print(userInputDataRaw.index)

In [ ]:
# Convert back to timedelta
timeSeriesData_BIGraw['timestamp'] = pd.to_timedelta(timeSeriesData_BIGraw['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIGraw ["datetime_timestamp"]= timeSeriesData_BIGraw['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment']
columns_timedelta = ['time taken total','time taken before insertion',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']
# Ensure target columns are of object type before assignment
userInputDataRaw[columns_datetime] = userInputDataRaw[columns_datetime].astype('object')
userInputDataRaw[columns_timedelta] = userInputDataRaw[columns_timedelta].astype('object')

userInputDataRaw.loc[:,columns_datetime] = userInputDataRaw.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputDataRaw.loc[:,columns_timedelta] = userInputDataRaw.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
timeSeriesData_BIG = timeSeriesData_BIGraw.copy()
userInputData = userInputDataRaw.copy()

In [ ]:
userInputData

In [ ]:
userInputData["room"].unique()

In [ ]:
#keep the data from the last set experiments made that have the 3 sensors in a triangle shape,they have 16 particular points in the space d
room_mask = userInputData["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'])
open_door_mask = userInputData["are-doors-opened"] != "on"

mask = room_mask  & open_door_mask
userInputData = userInputData.loc[mask]
#grab all the data that are contained in those experiments
timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

In [ ]:
userInputData

In [ ]:
timeSeriesData_BIG

In [ ]:
column_to_transform = "x-y axis"
userInputData.loc[:,column_to_transform] = userInputData.loc[:,column_to_transform].apply(tuple)

In [ ]:
#from the experiments with door open, drop the positions which doesn't fit to the 16 source position we are going to check the model
#also drop the experiments with the None values
axis_list = [(None, None), (-2.15, 2.0), (-1.35, 2.0), (-1.35, 1.5),
       (-1.35, 1.0), (-1.85, 1.0), (-1.85, 1.5)]
axis_mask = ~userInputData["x-y axis"].isin(axis_list)

mask = axis_mask 
userInputData = userInputData.loc[mask]
#grab all the data that are contained in those experiments
timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

In [ ]:
userInputData

In [ ]:

euclidian_distance_columns = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                               for id in range(3) ]

In [ ]:
np.sort(userInputData[euclidian_distance_columns[0]].unique())

In [ ]:
np.sort(userInputData[euclidian_distance_columns[1]].unique())

In [ ]:
np.sort(userInputData[euclidian_distance_columns[2]].unique())

In [ ]:
userInputData

In [ ]:
timeSeriesData_BIG

In [ ]:
userInputData.loc[userInputData["are-doors-opened"]=="on"].shape

In [ ]:
userInputData.loc[userInputData["are-doors-opened"]!="on"].shape

In [ ]:
max_before= -30
max_after = 299

column_to_check = "time taken before insertion (seconds)-capped"
mask = userInputData[column_to_check] >max_before
print(f"{column_to_check} < {max_before}:\n {userInputData.loc[mask,column_to_check]}")

column_to_check = "time taken after insertion (seconds)-capped"
mask = userInputData[column_to_check] < max_after
print(f"{column_to_check} < {max_after}:\n {userInputData.loc[mask,column_to_check]}")
print(f"userInputData rows {userInputData.shape[0]}")

In [ ]:
max_before= -30
max_after = 299

mask_before = (userInputData["time taken before insertion (seconds)-capped"] == max_before)
max_after = (userInputData["time taken after insertion (seconds)-capped"] == max_after)
mask = mask_before & max_after

userInputData = userInputData.loc[mask,:].copy()
print(f"userInputData rows {userInputData.shape[0]}")
timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

In [ ]:
userInputData.columns

In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
sensors_euclidian_distance_columns = ["Euclidian distance to Id="+str(id) + ":BME680:breathVocEquivalent"
                                    for id in range(3)]

userInputData.loc[:,sensors_position_columns] = userInputData.loc[:,sensors_position_columns].applymap(lambda lst: [round(x,2) for x in lst])
userInputData.loc[:,sensors_euclidian_distance_columns] = userInputData.loc[:,sensors_euclidian_distance_columns].round(2)


In [ ]:
userInputData[sensors_position_columns] 

In [ ]:
userInputData[sensors_euclidian_distance_columns]

In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
column_to_transform = sensors_position_columns
userInputData.loc[:,column_to_transform] = userInputData.loc[:,column_to_transform].apply(tuple)

## CHECK IF THE THREE SENSORS HAVE A COMMON POINT

In [ ]:
from scipy.optimize import least_squares

def trilateration_least_squares(points, radii):
    def fun(p):
        x, y = p
        return [np.hypot(x - px, y - py) - r for (px, py), r in zip(points, radii)]
    res = least_squares(fun, x0=[np.mean([px for px,_ in points]), np.mean([py for _,py in points])])
    x, y = res.x
    # Check residuals
    if np.all(np.abs(fun([x, y])) < 1e-2):
        return True, (x, y)
    else:
        return False, (x, y)

In [ ]:


def common_circle_point_linear_apply_wrapper(row):
    sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
    sensors_euclidian_distance_columns = ["Euclidian distance to Id="+str(id) + ":BME680:breathVocEquivalent"
                                    for id in range(3)]

    points =row[sensors_position_columns]
    radii = row[sensors_euclidian_distance_columns]
     # Extract points: make sure each entry is a tuple (x, y)
    points = [tuple(row[col]) for col in sensors_position_columns]
    print(f"points{points}")
    # Extract radii as floats
    radii = [float(row[col]) for col in sensors_euclidian_distance_columns]
    found, point = trilateration_least_squares(points,radii)
    return found, point

In [ ]:

userInputData.apply(common_circle_point_linear_apply_wrapper,axis=1)


## PRINT DATA PER SENSOR

### Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}
for index,data in dict_of_timeseries.items():
    dict_of_timeseries[index] = dict_of_timeseries[index].set_index("seconds",drop=False)

In [ ]:
def plot_position(userInputData,sample_row_of_the_group):
    plt.figure(figsize=(6, 6))  
    position_of_sensors = userInputData.iloc[-1]
    all_positions = userInputData.loc[:, ["x axis", "y axis"]]
    # Extra points
    extra_positions = np.array([
        [position_of_sensors["position of Id=0:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=0:BME680:breathVocEquivalent-y axis"]],
    
        [position_of_sensors["position of Id=1:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=1:BME680:breathVocEquivalent-y axis"]],
        [position_of_sensors["position of Id=2:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=2:BME680:breathVocEquivalent-y axis"]]
    ])
    extra_ids = ["id0","id1", "id2"]
    
    
    room_length = 4.0
    room_width = 3.25
    
    
    # Create scatterplot of the sources of all the particular setup
    #sns.scatterplot(x=positions[:,0], y=positions[:,1])
    sns.scatterplot(data=all_positions, x="x axis", y="y axis", color='blue', s=100, label='User Input Data')
    
    
    
    # Add the positions of sensors
    sns.scatterplot(x=extra_positions[:,0], y=extra_positions[:,1], color='red', s=100)
    x_sensor_highlight = sample_row_of_the_group["x axis"]
    y_sensor_highlight = sample_row_of_the_group["y axis"]
    # Plot a hollow circle around it
    plt.scatter(x_sensor_highlight, y_sensor_highlight, s=500, facecolors='none', edgecolors='green', linewidths=2, label='Highlighted point')  
    # Draw lines and annotate distances
    distances_from_sensors = (
        sample_row_of_the_group["Euclidian distance to Id=0:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=1:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=2:BME680:breathVocEquivalent"]
    )
    
    for i, (x, y) in enumerate(extra_positions):
        plt.plot([x_sensor_highlight, x], [y_sensor_highlight, y], color='red', linewidth=0.7, alpha=0.7)
        
        if distances_from_sensors is not None:
            # Midpoint of the line for annotation
            mid_x = (x_sensor_highlight + x) / 2
            mid_y = (y_sensor_highlight + y) / 2
            plt.text(mid_x, mid_y, f"{distances_from_sensors[i]:.2f}", color='red', fontsize=8, ha='center', va='center',
                         bbox=dict(facecolor='white', edgecolor='none', alpha=0.6, pad=1))
    # Annotate extra points with their IDs
    for i, (x, y) in enumerate(extra_positions):
        plt.text(x, y, extra_ids[i], fontsize=12, ha='right', va='bottom', color='red')
    
    
    # Set axis limits
    plt.xlim(-room_width, 0)
    plt.ylim(0, room_length)
    
    # Add grid
    plt.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)
    # Smaller legend
    plt.legend(fontsize=8, markerscale=0.8, labelspacing=0.4)

    plt.show()

In [ ]:
def printDataBasedOnDate(column_to_print,userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping):
    
    column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"green","Id=2:BME680:breathVocEquivalent":"yellow"}
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        timeSeriesData_BIG_copy = timeSeriesData_BIG.copy() 

        if ("position"  in type_of_other_grouping):
            sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
            
            plot_position(userInputData,sample_row_of_the_group)       
        print(f"group_name {group_name}")
        print(f"indexes_of_the_group {indexes_of_the_group}")
        data = timeSeriesData_BIG_copy.loc[timeSeriesData_BIG_copy["keys"].isin(indexes_of_the_group),:]  
      # Create relplot
        g = sns.relplot(
            data=data,
            x="seconds",
            y=column_to_print,
            hue="sensors",
            col="keys",        # separate subplot per key
            kind="line",
            col_wrap=3, 
                height=7,    # default = 5
            aspect=1, # width = height × aspect (so 6 × 1.5 = 9 inches wide per subplot
            palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  
            linewidth=2,
           facet_kws={
            "sharex": False,
            "sharey": False       
    
        })
        

        # >>> ADD THIS PART <<<
        for ax in g.axes.flat:
            ax.xaxis.set_major_locator(MultipleLocator(30))
            ax.xaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(True, which='both', linestyle=':', linewidth=0.5)
            
 
        
    # Get the horizontal and  vertical line position for this experiment
        for key_value, ax in g.axes_dict.items():
           
                #value to show the time that source is inserted
          
            userInputDataRow = userInputData.loc[key_value,:]
        #    x_position = f"side-right-wall {userInputDataRow['side-right-wall']},side-left-wall {userInputDataRow['side-left-wall']} \n"
        #    y_position = f"front-wall {userInputDataRow['front-wall']},back-wall {userInputDataRow['back-wall']} \n"
            
            euclidian_distances = (
                                  f"distance from Id0 sensor {userInputDataRow['Euclidian distance to Id=0:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id1 sensor {userInputDataRow['Euclidian distance to Id=1:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id2 sensor {userInputDataRow['Euclidian distance to Id=2:BME680:breathVocEquivalent']}\n",
            )
            subtitle=  (
                        f"At experiment with key {key_value}\n datetime:{userInputDataRow['actual timestamp StartingExperiment']}\n", 
                        f"experimentState:{userInputDataRow['experimentState']}\n",
                        f"x-axis: {userInputDataRow['x axis']} , y-axis: {userInputDataRow['y axis']}\n"
            )
            if ("distance"  in type_of_other_grouping):
               subtitle = subtitle + "\n"+euclidian_distances  
            ax.set_title(subtitle, fontsize=9)   
            g.fig.suptitle(f"Group: {group_name}", fontsize=16)
        
            g.fig.subplots_adjust(
                    top=0.75,   # space for overall title
                    wspace=0.2, # horizontal space between subplots
                    hspace=0.3 # vertical space between subplots
                   )

        plt.show()   
             

In [ ]:
def plot_all_positions(userInputData):
    room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
        plot_position(userInputData,sample_row_of_the_group)      
        
plot_all_positions(userInputData)

sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState",*euclidian_distances_columns]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","position","distance"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState",*euclidian_distances_columns]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","position","distance"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState"]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","date of experiment"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

## MODEL TRAINING

### CREATE THE X AND Y ARRAYS

In [ ]:

df_filtered = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
dfs_by_sensor = {
    sensor: g
    for sensor, g in df_filtered.groupby("sensors")
}


In [ ]:
dfs_by_sensor

In [ ]:
dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].columns

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()

euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
userInputData.loc[:,euclidian_distances_columns] = userInputData.loc[:,euclidian_distances_columns].round(2)

In [ ]:
print(userInputData["x-y axis"].unique())

In [ ]:
columns_to_keep = ["VOC rolling average"]

key_to_grab_size = userInputData.index[0]
mask = dfs_by_sensor['Id=0:BME680:breathVocEquivalent']["keys"] == key_to_grab_size

sample_data = dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].loc[mask,columns_to_keep]
print(sample_data.shape[0])


In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                        for id in range(3)]
sensors_euclidian_distance_columns = ["Euclidian distance to Id="+str(id) + ":BME680:breathVocEquivalent"
                                for id in range(3)]
print(sensors_position_columns)
print(sensors_euclidian_distance_columns)

In [ ]:
sensors_euclidian_distance_columns = ["Euclidian distance to Id="+str(id) + ":BME680:breathVocEquivalent"
                                for id in range(3)]

columns_to_create_X_data = "VOC rolling average"

column_to_create_Y_data = sensors_euclidian_distance_columns

sensors = timeSeriesData_BIG["sensors"].unique()
X ={}
Y ={}
true_positions = {}
#grab the number of features

key_to_grab_size = userInputData.index[0]
mask = dfs_by_sensor['Id=0:BME680:breathVocEquivalent']["keys"] == key_to_grab_size

sample_data = dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].loc[mask,columns_to_create_X_data]
X_subset_columns_size = sample_data.shape[0]

dict_flattened_arrays_per_sensor_per_distance ={}
for sensor,column_to_create_Y_data in zip(sensors,column_to_create_Y_data):
    X[sensor] = np.empty((0, X_subset_columns_size))
    Y[sensor] = np.empty((0, 1))
    true_positions[sensor] = np.empty((0, 2))
    dict_flattened_arrays_per_sensor_per_distance[sensor] = {}
    
    for euclidian_distance,experiments in userInputData.groupby(column_to_create_Y_data):
        
        rows_size = experiments.shape[0]
        true_positions_subset =  np.empty((rows_size,2))
        
        Y_distance_subset =np.empty((rows_size,1))
        X_distance_subset = np.empty((rows_size,X_subset_columns_size))

        Y_distance_subset[:] = euclidian_distance
        true_positions_subset[:,0] = experiments.iloc[0]["x-y axis"][0]
        true_positions_subset[:,1] = experiments.iloc[0]["x-y axis"][1]  
        
        Y[sensor] = np.vstack((Y[sensor],Y_distance_subset)) 
        true_positions[sensor] = np.vstack((true_positions[sensor],true_positions_subset)) 
             
        for array_index,experiment_index in enumerate(experiments.index):
          
            mask = dfs_by_sensor[sensor]["keys"]== experiment_index
            flatten_array = dfs_by_sensor[sensor].loc[mask,columns_to_create_X_data].to_numpy().reshape(1, -1)
            
            X_distance_subset[array_index,:] =   flatten_array
            
        X[sensor] = np.vstack((X[sensor],X_distance_subset))   
            



In [ ]:
X

In [ ]:
Y

In [ ]:
true_positions

In [ ]:
X.keys()

In [ ]:
sensor_names = timeSeriesData_BIG["sensors"].unique()
sensor_names = np.sort(sensor_names)
sensor_names

## DISCOVER POTENTIAL DIMENSIONALITY REDUCTION

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:


sensor_names = ['Id=0:BME680:breathVocEquivalent','Id=1:BME680:breathVocEquivalent','Id=2:BME680:breathVocEquivalent']


for sensor_name in sensor_names:
    print(sensor_name)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X[sensor_name])
    print(scaler.get_params())
    # Step 2: Apply PCA
    pca = PCA()
    X_pca = pca.fit_transform(X_scaled)
    
    # Step 3: Explained variance
    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)
    
    # Only display first 10 components max
    max_components = min(10, len(explained_variance))
    ev_to_plot = explained_variance[:max_components]
    cum_to_plot = cumulative_variance[:max_components]
    
    # Step 4: Bar plot of cumulative explained variance
    plt.figure(figsize=(8,5))
    plt.bar(range(1, max_components + 1), cum_to_plot)
    plt.xlabel('Number of PCA components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Cumulative explained variance (first 10 components)')
    plt.grid(True, axis='y')
    plt.show()
    
    # Step 5: Optimal number of components for ~90% variance
    optimal_components = np.argmax(cumulative_variance >= 0.90) + 1
    print("Optimal number of components to explain ~90% variance:", optimal_components)


## DISCOVER AND SEARCH STATISTICAL FEATURES

In [ ]:
X_binned = {}
for sensor,sensor_X in X.items(): 
    rows_size, columns_size = sensor_X.shape
    
    X_binned[sensor] = sensor_X.reshape(rows_size, columns_size // 15, 15)
    
print(X_binned)    

In [ ]:
for sensor_name in X_binned:
    print(X_binned[sensor_name].shape)

In [ ]:
X_mean_and_variance ={}
for sensor,sensor_X_binned  in X_binned.items(): 
    X_mean_and_variance[sensor] = np.empty((X_binned[sensor].shape[0],X_binned[sensor].shape[1],2))

for sensor,sensor_X_binned in X_binned.items(): 
    #enumerate for each 2 dimension matrix that contains each bin
    for i,bin_data in enumerate(sensor_X_binned):
        X_mean_and_variance[sensor][i,:,0] = bin_data.mean(axis=1)
        X_mean_and_variance[sensor][i,:,1] = bin_data.var(axis=1)        

In [ ]:
for sensor_name in X_mean_and_variance:
    print(X_mean_and_variance[sensor_name].shape)

In [ ]:
s = sensor_names[0]
X_mean_and_variance[s]

In [ ]:
X_mean_and_variance_df = {}
for sensor_name,data in X_mean_and_variance.items():

    # Generate time window labels automatically
    window_size = 15
    num_windows = data.shape[1]
    
    # Starting at -30 seconds as in your original example
    start_time = -30
    time_titles = [f"{start_time + i*window_size}:{start_time + (i+1)*window_size - 1} seconds" 
                   for i in range(num_windows)]
    
    # Reshape data to 2D for DataFrame creation: (98*22, 2)
    flat_data = data.reshape(-1, 2)
    
    # Create MultiIndex: outer index = 0..97, inner index = time_titles repeated
    outer_index = np.repeat(np.arange(data.shape[0]), num_windows)
    inner_index = np.tile(time_titles, data.shape[0])
    multi_index = pd.MultiIndex.from_arrays([outer_index, inner_index], names=['sample', 'time_window'])
    
    # Create DataFrame
    X_mean_and_variance_df[sensor_name] = pd.DataFrame(flat_data, index=multi_index, columns=['mean', 'variance'])



In [ ]:
X_mean_and_variance_df

In [ ]:
#add y values
for sensor_name,df in X_mean_and_variance_df.items():
    
    assert len(Y[sensor_name]) == df.index.get_level_values('sample').nunique()
    sample_to_Y = {sample: float(y[0]) for sample, y in enumerate(Y[sensor_name])}
    # Add new column 'Y' based on the outer sample index
    df['Y'] = df.index.get_level_values('sample').map(sample_to_Y)

In [ ]:
X_mean_and_variance_df

In [ ]:

for sensor_name,df in X_mean_and_variance_df.items():
    X_mean_and_variance_df[sensor_name] = df.reset_index()


In [ ]:
# make the multi index columns and make the time window categorical
#set the order
time_window_order = [
    "-30:-16 seconds", "-15:-1 seconds", "0:14 seconds", "15:29 seconds",
    "30:44 seconds", "45:59 seconds", "60:74 seconds", "75:89 seconds",
    "90:104 seconds", "105:119 seconds", "120:134 seconds", "135:149 seconds",
    "150:164 seconds", "165:179 seconds", "180:194 seconds", "195:209 seconds",
    "210:224 seconds", "225:239 seconds", "240:254 seconds", "255:269 seconds",
    "270:284 seconds", "285:299 seconds"
]    
for sensor_name,df in X_mean_and_variance_df.items():
    X_mean_and_variance_df[sensor_name]['time_window'] = pd.Categorical(df['time_window'],
                                        categories=time_window_order,
                                        ordered=True)

In [ ]:
X_mean_and_variance_df

In [ ]:
# remove big values 
outliers = {}
for sensor_name,df in X_mean_and_variance_df.items():

     # Compute Q1 and Q3 per variable
    Q1 = df['variance'].quantile(0.25)
    Q3 = df['variance'].quantile(0.75)
    IQR = Q3 - Q1
    
    # Compute bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Find outlier rows
    outliers[sensor_name] = df[(df['variance'] > upper_bound)]
    
            

In [ ]:
outliers

In [ ]:
# remove big values 
no_outliers = {}
for sensor_name,df in X_mean_and_variance_df.items():

     # Compute Q1 and Q3 per variable
    Q1 = df['variance'].quantile(0.25)
    Q3 = df['variance'].quantile(0.75)
    IQR = Q3 - Q1
    
    # Compute bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Find outlier rows
    no_outliers[sensor_name] = df[df['variance'] <= upper_bound]
    


In [ ]:
no_outliers

In [ ]:
for sensor_name,data in X_mean_and_variance_df.items():
    print(sensor_name)
    print(data.shape)

In [ ]:
for sensor_name,data in no_outliers.items():
    print(sensor_name)

    print(data.shape)    

In [ ]:
no_outliers

In [ ]:
# take percentage removed from rows
column = "time_window"
for (sensor_name,data_whole),data_removed in zip(X_mean_and_variance_df.items(),no_outliers.values()):
    print(sensor_name)
    
    percentange = data_removed.shape[0] / data_whole.shape[0]
    print(f"percentange-total:{percentange}")

    for (time_window,data_whole_time_window),(time_window,data_removed_time_window) in zip(data_whole.groupby(column),data_removed.groupby(column)):
        
        percentage_time_window =  data_removed_time_window.shape[0] / data_whole_time_window.shape[0] 
        print(f"for time window:{time_window},we have keep {percentage_time_window} of the original data")


In [ ]:
#add columns

In [ ]:
def printExtractedFeatures(features_extracted):
    for sensor_name,data in features_extracted.items():
            # data is your numpy array of shape (98, 22, 2)
        # Example placeholder; REMOVE this line in your notebook:
        # data = np.random.rand(98, 22, 2)
            
        df_plot = data.reset_index()
       # print(df_plot)
        # Optionally, create a numeric 'group' from time_window for coloring
       # df_plot['group'] = df_plot['time_window'].astype('category').cat.codes
        
        # Plot
        plt.figure(figsize=(16, 10))
        flare_palette = sns.color_palette("flare", n_colors=(df_plot['time_window'].nunique()).shape[0])
        
        sns.scatterplot(
            data=df_plot,
            x="mean",
            y="variance",
            hue="time_window",
            palette=flare_palette,
            s=40,
            edgecolor="none"
        )
        
        plt.title(f"{sensor_name},Scatterplot of mean vs variance (MultiIndex DataFrame)", fontsize=18)
        plt.xlabel("Mean")
        plt.ylabel("Variance")
        
        # Put legend outside
        plt.legend(
            title="Time Window",
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
            borderaxespad=0.
        )
        
        plt.tight_layout()
        plt.show()

In [ ]:
def printExtractedFeaturesBox(features_extracted):
    for sensor_name,df in features_extracted.items():
        plt.figure(figsize=(16, 8))
       
        sns.boxplot(
            data=df,
            x='time_window',
            y='variance',
           # hue="Y",

          #  palette='flare'
        )
        plt.xticks(rotation=45)
        plt.title(f"{sensor_name}:Violin Plot of Variance by Time Window", fontsize=18)
        plt.xlabel("Time Window")
        plt.ylabel("Variance")
        plt.tight_layout()
        plt.show()


In [ ]:
def printExtractedFeaturesViolin(features_extracted):
    for sensor_name,df in features_extracted.items():
        plt.figure(figsize=(16, 8))
        #flare_palette = sns.color_palette("flare", n_colors=(df_plot['Y'].nunique()))
   
        sns.violinplot(
            data=df,
            x='time_window',
            y='variance',
           # palette='flare',
            inner='quartile'  # shows median and quartiles
        )
        plt.xticks(rotation=45)
        plt.title(f"{sensor_name}:Violin Plot of Variance by Time Window", fontsize=18)
        plt.xlabel("Time Window")
        plt.ylabel("Variance")
        plt.tight_layout()
        plt.show()


In [ ]:
def printExtractedFeaturesBoxPerY(features_extracted,name="no outliers"):
    for sensor_name, df in features_extracted.items():
     

        # Use Seaborn catplot with row = Y to create separate plots per unique Y
        sharey_flag = True
        if name == "outliers":
            sharey_flag = False
        g = sns.catplot(
            data=df,
            x='time_window',
            y='variance',
            kind='box',    # violin plot
            row='Y',          # facet rows by Y
            sharey=sharey_flag,
            height=4,
            aspect=3,
            palette='flare',
        )
           
        for ax in g.axes.flat:
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        g.set_xticklabels(rotation=45)
        g.set_axis_labels("Time Window", "Variance")
        g.fig.suptitle(f"{sensor_name}: Boxplot of Variance by Time Window grouped by Y", fontsize=18)
        g.fig.subplots_adjust(top=0.92)  # adjust space for the suptitle

        plt.show()

In [ ]:
printExtractedFeaturesBox(X_mean_and_variance_df)

In [ ]:
printExtractedFeaturesBox(outliers)

In [ ]:
printExtractedFeaturesBox(no_outliers)

In [ ]:
printExtractedFeaturesViolin(no_outliers)

In [ ]:
printExtractedFeaturesBoxPerY(X_mean_and_variance_df)

In [ ]:
printExtractedFeaturesBoxPerY(no_outliers)

In [ ]:
printExtractedFeaturesBoxPerY(outliers,"outliers")

In [ ]:
no_outliers['Id=2:BME680:breathVocEquivalent']

In [ ]:
def barplot(mean_per_window,statistic):
    for sensor_name,df in mean_per_window.items():

        plt.figure(figsize=(16, 6))
        sns.barplot(
            data=df,
            x='time_window',
            y=statistic,
            palette='flare'
        )
        
        plt.xticks(rotation=45)
        plt.title(f"{sensor_name}:Mean {statistic} per Time Window", fontsize=18)
        plt.xlabel("Time Window")
        plt.ylabel(f"Mean {statistic}")
        plt.tight_layout()
        plt.show()

def create_bar_plot(all_data,statistic):   
    mean_per_window = {}
    for sensor_name,df in all_data.items():
        mean_per_window[sensor_name] = df.groupby('time_window')[statistic].mean().reset_index()
    barplot(mean_per_window,statistic)

In [ ]:
create_bar_plot(no_outliers,"mean")

In [ ]:
create_bar_plot(no_outliers,"variance")

In [ ]:
create_bar_plot(X_mean_and_variance_df,"mean")

In [ ]:
create_bar_plot(X_mean_and_variance_df,"variance")

## FEATURE EXTRACTION

Follow the first steps of the previous chapter, but merge the first 3 windows

In [ ]:
X_binned = {}
for sensor,sensor_X in X.items(): 
    rows_size, columns_size = sensor_X.shape
    
    X_binned[sensor] = sensor_X.reshape(rows_size, columns_size // 15, 15)
    
print(X_binned)    

In [ ]:
X_mean_and_variance = {}

for sensor, sensor_X in X_binned.items():
    
    n_samples, n_windows, n_points = sensor_X.shape
    
    # after merging first 3 windows → 1 big window + 19 remaining = 20
    new_windows = 1 + (n_windows - 3)
    X_mean_and_variance[sensor] = np.empty((n_samples, new_windows, 2))
    
    for i in range(n_samples):
        data = sensor_X[i]   # shape (22, 15)
        
        # --- MERGE FIRST THREE WINDOWS ---
        merged = data[0:3].reshape(-1)  # flatten 3×15 into 45 samples
        
        X_mean_and_variance[sensor][i, 0, 0] = merged.mean()
        X_mean_and_variance[sensor][i, 0, 1] = merged.var()
        
        # --- PROCESS THE REMAINING WINDOWS AS BEFORE ---
        # windows 3–21 → indices 1–19 in new array
        X_mean_and_variance[sensor][i, 1:, 0] = data[3:].mean(axis=1)
        X_mean_and_variance[sensor][i, 1:, 1] = data[3:].var(axis=1)


In [ ]:
X_mean_and_variance

In [ ]:
for sensor_name in X_mean_and_variance:
    print(X_mean_and_variance[sensor_name].shape)

Change the values of the original X to values of the X_mean_and_variance, in order to use the code above as it is

In [ ]:
X_TEMP = {}
for sensor_name,data in X_mean_and_variance.items():
    experiments_size = data.shape[0]
    X_TEMP[sensor_name] = data.reshape(experiments_size,-1)

In [ ]:
X_TEMP

In [ ]:
for sensor_name in X_TEMP:
    print(X_TEMP[sensor_name].shape)

In [ ]:
X = X_TEMP

In [ ]:
X

## DISCOVER POTENTIAL DIMENSIONALITY REDUCTION FOR FEATURE EXTRACTION

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:


sensor_names = ['Id=0:BME680:breathVocEquivalent','Id=1:BME680:breathVocEquivalent','Id=2:BME680:breathVocEquivalent']


for sensor_name in sensor_names:
    print(sensor_name)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X[sensor_name])
    print(scaler.get_params())
    # Step 2: Apply PCA
    pca = PCA()
    X_pca = pca.fit_transform(X_scaled)
    
    # Step 3: Explained variance
    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)
    
    # Only display first 10 components max
    max_components = min(10, len(explained_variance))
    ev_to_plot = explained_variance[:max_components]
    cum_to_plot = cumulative_variance[:max_components]
    
    # Step 4: Bar plot of cumulative explained variance
    plt.figure(figsize=(8,5))
    plt.bar(range(1, max_components + 1), cum_to_plot)
    plt.xlabel('Number of PCA components')
    plt.ylabel('Cumulative explained variance')
    plt.title('Cumulative explained variance (first 10 components)')
    plt.grid(True, axis='y')
    plt.show()
    
    # Step 5: Optimal number of components for ~90% variance
    optimal_components = np.argmax(cumulative_variance >= 0.90) + 1
    print("Optimal number of components to explain ~90% variance:", optimal_components)


### SEARCH BEST MODEL OF EACH INDIVIDUAL SENSOR'S X AND Y AXIS

In [ ]:
# Required imports
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score


In [ ]:

# 1. Linear Regression (no hyperparameters to tune, just for completeness)
lr = LinearRegression()
lr_params = {}  # No parameters for simple LinearRegression

# 2. Ridge Regression
ridge = Ridge()
ridge_params = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

# 3. Lasso Regression
lasso = Lasso()
lasso_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
}

# 4. ElasticNet
elastic = ElasticNet()
elastic_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

# 5. Support Vector Regression
svr = SVR()
svr_params = {
    'kernel':["rbf"],
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.001, 0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"]
}


# 7. Random Forest Regression
rf = RandomForestRegressor()
rf_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 8. Gradient Boosting Regression
gbr = GradientBoostingRegressor()
gbr_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [ 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

models = {
    'LinearRegression': (lr, lr_params),
    'Ridge': (ridge, ridge_params),
    'Lasso': (lasso, lasso_params),
    'ElasticNet': (elastic, elastic_params),
    'SVR': (svr, svr_params),
    'RandomForest': (rf, rf_params),
    #'GradientBoosting': (gbr, gbr_params)
}

PCA__n_components = [3,4,5,6,8,10]


In [ ]:

def run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params,PCA__n_components):
        estimators_with_no_scaling_need = ['RandomForest','DecisionTree','GradientBoosting']

        results = {}
        if name in estimators_with_no_scaling_need:
             pipe = Pipeline([
                 ('model', model)
             ]) 
        else:
             pipe = Pipeline([
                 ('scaler', StandardScaler()),
                 ('PCA', PCA()),
                 ('model', model)
             ])
        # Build a correct param grid:
        param_grid = {
            **{'model__' + k: v for k, v in params.items()}
        }
        if name not in estimators_with_no_scaling_need:
            param_grid['PCA__n_components'] = PCA__n_components
            
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=cv_number,
            n_jobs=-1,
            scoring='neg_mean_squared_error',
            verbose=2
        )
     
        grid.fit(X_train, y_train)
    
        results["name"]  = name
        results["model"] = grid.best_estimator_
        results["parameters"] = grid.best_params_
        results["score"] = grid.best_score_
    
        return results

def run_grid_search_find_optimal_model_per_sensor(X_train, y_train,cv_number,verbose,models,PCA__n_components):
    #put a score way worse than abest_score_nything we should have
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {}    
    }
    for name, (model, params) in models.items():
        print(f"Running GridSearchCV for {name}...")

        results = run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params,PCA__n_components)
        print(results)

        if (results["score"] >  best_result["score"]) :
            best_result["name"] = results["name"]
            print(best_result["name"])
            best_result["score"] = results["score"]
            print(best_result["score"])
            best_result["model"] = results["model"]
            print(best_result["model"])
            best_result["parameters"] = results["parameters"]
            print(best_result["parameters"])
    return best_result

In [ ]:
sensor_names = [
    "Id=0:BME680:breathVocEquivalent",
    "Id=1:BME680:breathVocEquivalent",
    "Id=2:BME680:breathVocEquivalent"    
]

test_size = 0.2         # number of data taken
random_state = 42         # predefined random state for reproducibility
cv_number = 5
verbose =2

best_result_per_sensor ={}
for sensor_name in sensor_names:
    X_arr = np.asarray(X[sensor_name])
    Y_arr_full = np.asarray(Y[sensor_name])
    Y_arr_Current_Axis = 0
    Y_arr = Y_arr_full[:,Y_arr_Current_Axis]

    # --- Train/test split ---
    # --- Train/test split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X_arr, Y_arr,
        test_size=0.16,
        stratify=Y_arr,
        random_state=42
    )
    best_result_per_sensor[sensor_name] = run_grid_search_find_optimal_model_per_sensor(X_train,y_train,cv_number,verbose,models,PCA__n_components)

In [ ]:
best_result_per_sensor

### SEE RESULTS

In [ ]:
def plot_pred_vs_actual(y_test, y_pred,sensor_name,AXIS):
    # --- Plot: predicted vs actual ---
    mse = mean_squared_error(y_test, y_pred)
    
    r2  = r2_score(y_test, y_pred)
    #check if a numpy array is 1D,if yes,make it 2D with one n,1
    if y_test.ndim == 1:
       y_test =  y_test.reshape(-1,1)
    if y_pred.ndim == 1:
       y_pred =  y_pred.reshape(-1,1)   

    EucDis = np.linalg.norm(y_test - y_pred,axis=1).mean()
    print(f"MSE on test set: {mse:.4f}")
    
    print(f"R^2 on test set: {r2:.4f}")

    print(f"Euclidian distance median value {EucDis:.4f}")
  

    if (y_test.shape[1] == 1) and (y_pred.shape[1] == 1):
        plt.figure(figsize=(7,6))
        plt.scatter(y_test, y_pred, s=50)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle='--')
        plt.xlabel("True target")
        plt.ylabel("Predicted target")
        plt.title(f"{sensor_name} {AXIS} predictions vs actual")
        plt.grid(True)
        plt.show()

    else:
        # --- Plot ---
        plt.scatter(y_test[:,0], y_test[:,1], label="True", alpha=0.6)
        plt.scatter(y_pred[:,0], y_pred[:,1], label="Predicted", alpha=0.6)
        plt.legend()
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.title("True vs Predicted Positions")
        plt.grid(True)
        plt.show()


In [ ]:
#PREDICT AND SEE INDIVIDUAL RESULTS
predict_Y = {}


for sensor_name in sensor_names:
    X_arr = np.asarray(X[sensor_name])
    Y_arr = np.asarray(Y[sensor_name])
    X_train, X_test, y_train, y_test = train_test_split(
        X_arr, Y_arr,
        test_size=0.16,
        stratify=Y_arr,
        random_state=42
    )
    #predict x axis
    predict_Y[sensor_name] = best_result_per_sensor[sensor_name]["model"].predict(X_test)

    plot_pred_vs_actual(y_test,predict_Y[sensor_name],sensor_name,"Euclidian distance")

   

## LOCATE THE PLACE OF THE SOURCE USING 

In [ ]:
#rewrite it again to be sure
from scipy.optimize import least_squares

def trilateration_least_squares(points, radii):
    def fun(p):
        x, y = p
        return [np.hypot(x - px, y - py) - r for (px, py), r in zip(points, radii)]
    res = least_squares(fun, x0=[np.mean([px for px,_ in points]), np.mean([py for _,py in points])])
    x, y = res.x
    # Check residuals
    if np.all(np.abs(fun([x, y])) < 1e-2):
        return True, (x, y)
    else:
        return False, (x, y)

In [ ]:
#Take the three points of the sensors
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
point_of_sensors = []
for sensors_position_column in sensors_position_columns:
    
    point_of_sensors.append(userInputData[sensors_position_column].iloc[0]) 

In [ ]:
point_of_sensors

In [ ]:
len(predict_Y)

In [ ]:
X_arr = np.asarray(X[sensor_name])
Y_arr = np.asarray(Y[sensor_name])
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, Y_arr,
    test_size=0.16,
    stratify=Y_arr,
    random_state=42
)
x_dimension = len(predict_Y)
y_dimension = y_test.shape[0]
predicted_euclidian_values = np.empty((y_dimension,x_dimension))
for i,sensor_name in enumerate(sensor_names):
    predicted_euclidian_values[:,i] =predict_Y[sensor_name]

In [ ]:
predicted_euclidian_values.shape

In [ ]:
predicted_euclidian_values

In [ ]:
point_of_sensors

In [ ]:
#get points true and false
points = np.empty((y_dimension,2))
bool_values =  np.empty((y_dimension,1))
for i,set_of_distances in enumerate(predicted_euclidian_values):
    bool_value,point = trilateration_least_squares(point_of_sensors,set_of_distances)
    points[i,:] = point
    print(bool_value)
    bool_values[i] = bool_value

In [ ]:
bool_values

In [ ]:
points

In [ ]:
X_arr = np.asarray(X['Id=2:BME680:breathVocEquivalent'])
Y_arr = np.asarray(true_positions['Id=2:BME680:breathVocEquivalent'])
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, Y_arr,
    test_size=0.16,
    stratify=Y_arr,
    random_state=42
)
plot_pred_vs_actual(y_test,points,sensor_names,"Point predicted")